In [1]:
!pip install streamlit==1.40.0 -q
!pip install streamlit-drawable-canvas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires watchdog<7.0.0,>=6.0.0, but you have watchdog 5.0.3 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.4 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import pytesseract
from PIL import Image
import pandas as pd
from streamlit_drawable_canvas import st_canvas
import re

st.set_page_config(page_title="Manual OCR Scanner", layout="wide")

st.title("🗞️ Newspaper OCR: Manual Zone Mode")
st.markdown("Draw boxes over columns in the order you want them read (e.g., Box 1 = Column 1).")

# Sidebar settings
st.sidebar.title("Settings")
mode = st.sidebar.radio("Selection Mode", ("Draw Boxes", "Edit/Move Boxes"))
drawing_mode = "rect" if mode == "Draw Boxes" else "transform"

uploaded_file = st.sidebar.file_uploader("Upload Newspaper Page", type=["jpg", "jpeg", "png"])

if uploaded_file:
    # Load and get image dimensions
    bg_image = Image.open(uploaded_file)
    w, h = bg_image.size

    # Calculate scale to fit screen (max width 800)
    display_width = 800
    scale = display_width / w
    display_height = int(h * scale)

    col1, col2 = st.columns([1.2, 0.8])

    with col1:
        st.subheader("1. Define Reading Zones")
        # Create a canvas component
        canvas_result = st_canvas(
            fill_color="rgba(0, 255, 0, 0.2)",  # Transparent green
            stroke_width=2,
            stroke_color="#00FF00",
            background_image=bg_image,
            update_streamlit=True, # Correct parameter name is update_freq
            width=display_width,
            height=display_height,
            drawing_mode=drawing_mode,
            key="canvas",
        )

    with col2:
        st.subheader("2. Extracted Text")

        # Process objects from canvas
        if canvas_result.json_data is not None:
            objects = canvas_result.json_data["objects"]
            if len(objects) > 0:
                full_text = ""

                # Sort objects by their creation or top/left if preferred
                # Here we follow the order they were drawn
                for i, obj in enumerate(objects):
                    # Only process rectangles
                    if obj["type"] == "rect":
                        # Map canvas coordinates back to original image size
                        left = obj["left"] / scale
                        top = obj["top"] / scale
                        width = (obj["width"] * obj["scaleX"]) / scale
                        height = (obj["height"] * obj["scaleY"]) / scale

                        # Crop and OCR
                        crop_box = (left, top, left + width, top + height)
                        cropped_img = bg_image.crop(crop_box)

                        with st.spinner(f"Scanning Zone {i+1}..."):
                            # Using --psm 4 (assume single column) for better results on newspaper crops
                            text = pytesseract.image_to_string(cropped_img, config='--psm 4')
                            full_text += f"{text.strip()}"

                st.text_area("OCR Output", full_text, height=500)
                st.download_button("Download All Text", full_text, file_name="newspaper_scan.txt")
            else:
                st.info("Draw a rectangle over a column to start.")
else:
    st.info("Please upload an image to begin.")

Overwriting app.py


In [ ]:
%%writefile app.py
import streamlit as st
import pytesseract
from PIL import Image
import pandas as pd
import re
from streamlit_drawable_canvas import st_canvas

def clean_ocr_text(text):
    # Rejoin hyphenated line breaks (e.g., "newspa-\nper" → "newspaper")
    text = re.sub(r'-\n', '', text)
    # Replace single newlines with a space (soft wraps)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r' +', ' ', text)
    # Preserve paragraph breaks (double newlines)
    text = re.sub(r'\n{2,}', '\n\n', text)
    return text.strip()

st.set_page_config(page_title="Manual OCR Scanner", layout="wide")

st.title("🗞️ Newspaper OCR: Manual Zone Mode")
st.markdown("Draw boxes over columns in the order you want them read (e.g., Box 1 = Column 1).")

# Sidebar settings
st.sidebar.title("Settings")
mode = st.sidebar.radio("Selection Mode", ("Draw Boxes", "Edit/Move Boxes"))
drawing_mode = "rect" if mode == "Draw Boxes" else "transform"

uploaded_file = st.sidebar.file_uploader("Upload Newspaper Page", type=["jpg", "jpeg", "png"])

if uploaded_file:
    # Load and get image dimensions
    bg_image = Image.open(uploaded_file)
    w, h = bg_image.size

    # Calculate scale to fit screen (max width 800)
    display_width = 800
    scale = display_width / w
    display_height = int(h * scale)

    col1, col2 = st.columns([1.2, 0.8])

    with col1:
        st.subheader("1. Define Reading Zones")
        canvas_result = st_canvas(
            fill_color="rgba(0, 255, 0, 0.2)",
            stroke_width=2,
            stroke_color="#00FF00",
            background_image=bg_image,
            update_streamlit=True,
            width=display_width,
            height=display_height,
            drawing_mode=drawing_mode,
            key="canvas",
        )

    with col2:
        st.subheader("2. Extracted Text")

        if canvas_result.json_data is not None:
            objects = canvas_result.json_data["objects"]
            if len(objects) > 0:
                full_text = ""

                for i, obj in enumerate(objects):
                    if obj["type"] == "rect":
                        left = max(0, int(obj["left"] / scale))
                        top = max(0, int(obj["top"] / scale))
                        right = min(w, int((obj["left"] + obj["width"] * obj["scaleX"]) / scale))
                        bottom = min(h, int((obj["top"] + obj["height"] * obj["scaleY"]) / scale))
                        crop_box = (left, top, right, bottom)

                        cropped_img = bg_image.crop(crop_box)

                        with st.spinner(f"Scanning Zone {i+1}..."):
                            try:
                                text = pytesseract.image_to_string(cropped_img, config='--psm 4')
                                full_text += f"{clean_ocr_text(text)}"
                            except pytesseract.TesseractNotFoundError:
                                st.error("Tesseract OCR engine not found. Please install it.")
                                st.stop()
                            except Exception as e:
                                full_text += f"[Error in zone {i+1}: {e}]\n\n"

                st.text_area("OCR Output", full_text, height=500)
                st.download_button("Download All Text", full_text, file_name="newspaper_scan.txt")
            else:
                st.info("Draw a rectangle over a column to start.")
else:
    st.info("Please upload an image to begin.")

Overwriting app.py


In [7]:
%%writefile app.py
import streamlit as st
import pytesseract
from PIL import Image, ImageDraw
import pandas as pd
import re
from streamlit_drawable_canvas import st_canvas

# ── helpers ──────────────────────────────────────────────────────────────────

def clean_ocr_text(text):
    text = re.sub(r'-\n', '', text)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n{2,}', '\n\n', text)
    return text.strip()

def detect_blocks(image):
    """Run Tesseract layout analysis and return canvas-ready rect objects."""
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    df = pd.DataFrame(data)
    df = df[df.conf > 0]
    df = df[df.text.str.strip() != ""]
    if df.empty:
        return []

    w, h = image.size
    scale = 800 / w

    rects = []
    for block_id in sorted(df['block_num'].unique()):
        b = df[df.block_num == block_id]
        x0 = int(b.left.min() * scale)
        y0 = int(b.top.min() * scale)
        x1 = int((b.left + b.width).max() * scale)
        y1 = int((b.top + b.height).max() * scale)
        bw = x1 - x0
        bh = y1 - y0
        if bw < 10 or bh < 10:
            continue
        rects.append({
            "type": "rect",
            "version": "4.4.0",
            "originX": "left",
            "originY": "top",
            "left": x0,
            "top": y0,
            "width": bw,
            "height": bh,
            "fill": "rgba(0,255,0,0.2)",
            "stroke": "#00FF00",
            "strokeWidth": 2,
            "scaleX": 1,
            "scaleY": 1,
            "angle": 0,
            "selectable": True,
        })
    return rects

def ocr_zones(image, objects, scale):
    w, h = image.size
    full_text = ""
    for i, obj in enumerate(objects):
        if obj.get("type") != "rect":
            continue
        left   = max(0, int(obj["left"] / scale))
        top    = max(0, int(obj["top"] / scale))
        right  = min(w, int((obj["left"] + obj["width"]  * obj["scaleX"]) / scale))
        bottom = min(h, int((obj["top"]  + obj["height"] * obj["scaleY"]) / scale))
        if right <= left or bottom <= top:
            continue
        cropped = image.crop((left, top, right, bottom))
        try:
            text = pytesseract.image_to_string(cropped, config='--psm 4')
            full_text += f"{i+1}\n{clean_ocr_text(text)}\n\n"
        except pytesseract.TesseractNotFoundError:
            st.error("Tesseract not found. Please install it.")
            st.stop()
        except Exception as e:
            full_text += f"[Error in zone {i+1}: {e}]\n\n"
    return full_text

# ── page setup ────────────────────────────────────────────────────────────────

st.set_page_config(page_title="Newspaper OCR Scanner", layout="wide")
st.title("🗞️ Newspaper OCR Scanner")
st.markdown("Zones are detected automatically. Edit, move, or redraw them before scanning.")

# ── sidebar ───────────────────────────────────────────────────────────────────

st.sidebar.title("Settings")
uploaded_file = st.sidebar.file_uploader("Upload Newspaper Page", type=["jpg", "jpeg", "png", "tiff"])
mode = st.sidebar.radio("Canvas Mode", ("Edit / Move Zones", "Draw New Zone", "Delete Mode"))

drawing_mode_map = {
    "Edit / Move Zones": "transform",
    "Draw New Zone":     "rect",
    "Delete Mode":       "transform",
}
drawing_mode = drawing_mode_map[mode]

# ── main ──────────────────────────────────────────────────────────────────────

if uploaded_file:
    image = Image.open(uploaded_file).convert("RGB")
    w, h  = image.size
    display_width  = 800
    scale          = display_width / w
    display_height = int(h * scale)

    # ── auto-detect once per uploaded file ───────────────────────────────────
    file_key = uploaded_file.name + str(uploaded_file.size)
    if st.session_state.get("file_key") != file_key:
        with st.spinner("Auto-detecting text zones…"):
            st.session_state["initial_objects"] = detect_blocks(image)
        st.session_state["file_key"] = file_key

    initial_drawing = {
        "version": "4.4.0",
        "objects": st.session_state.get("initial_objects", []),
    }

    col1, col2 = st.columns([1.2, 0.8])

    with col1:
        st.subheader("1. Review & Edit Zones")
        st.caption(f"{len(initial_drawing['objects'])} zones auto-detected — add, move, or delete before scanning.")

        canvas_result = st_canvas(
            fill_color="rgba(0, 255, 0, 0.2)",
            stroke_width=2,
            stroke_color="#00FF00",
            background_image=image,
            initial_drawing=initial_drawing,
            update_streamlit=True,
            width=display_width,
            height=display_height,
            drawing_mode=drawing_mode,
            key=f"canvas_{file_key}",
        )

    with col2:
        st.subheader("2. Extracted Text")

        if canvas_result.json_data is not None:
            objects = [o for o in canvas_result.json_data.get("objects", []) if o.get("type") == "rect"]
            zone_count = len(objects)

            if zone_count == 0:
                st.info("No zones on canvas. Draw at least one rectangle to scan.")
            else:
                st.caption(f"{zone_count} zone(s) ready.")
                if st.button("▶ Run OCR on All Zones", type="primary"):
                    # Sort zones left-to-right, top-to-bottom (newspaper reading order)
                    col_threshold = 80  # ~800px canvas / 10
                    objects_sorted = sorted(objects, key=lambda o: (
                        round(o["left"] / col_threshold),
                        o["top"]
                    ))
                    with st.spinner("Scanning…"):
                        full_text = ocr_zones(image, objects_sorted, scale)
                    st.session_state["ocr_result"] = full_text

        # Render result from session state so it survives canvas interactions
        if "ocr_result" in st.session_state:
            st.text_area("OCR Output", st.session_state["ocr_result"], height=500)
            st.download_button("⬇ Download Text", st.session_state["ocr_result"], file_name="newspaper_scan.txt")

else:
    st.info("Upload a newspaper image from the sidebar to begin.")

Overwriting app.py


In [ ]:
# 1. Install dependencies
!apt-get install -y tesseract-ocr
!pip install pytesseract streamlit pillow opencv-python-headless

# # 2. Get your public IP for the tunnel password
# print("\nCOPY THIS IP ADDRESS:")
# !curl ipv4.icanhazip.com
# print("\n")

# # 3. Run the app
# !streamlit run app.py & npx localtunnel --port 8501

In [ ]:
from google.colab.output import eval_js
print("Click the link below:")
print(eval_js("google.colab.kernel.proxyPort(8501)"))

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS=false --server.enableXsrfProtection=false

Click the link below:
https://8501-m-s-kkb-usw4c0-1goo2le00lrhd-c.us-west4-0.prod.colab.dev



  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

